In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import image, plotting
from nilearn.glm.second_level import SecondLevelModel
from nilearn.input_data import NiftiMasker
from tqdm import tqdm  # Progress bar


C:\Users\ducat\AppData\Local\Temp\ipykernel_13676\1902100879.py:6: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  from nilearn.input_data import NiftiMasker


In [2]:
from glob import glob

all_files = glob("../contrasts/smoothed_3mm/*")

all_files[:3]

['../contrasts/smoothed_3mm\\sub-11-25_+_35_over_5_+_15.nii.gz',
 '../contrasts/smoothed_3mm\\sub-11-35_+_45_over_5_+_15.nii.gz',
 '../contrasts/smoothed_3mm\\sub-11-45_+_55_over_5_+_15.nii.gz']

In [3]:
split = {v.split("-")[-1].split(".nii.gz")[0] for v in all_files}

contrast_list = sorted(list(split))

In [4]:
contrast_list

['25_+_35_over_5_+_15',
 '35_+_45_over_5_+_15',
 '45_+_55_over_5_+_15',
 '55_+_65_over_5_+_15',
 '65_+_75_over_5_+_15',
 '75_+_85_over_5_+_15',
 '85_+_95_over_5_+_15',
 'high_over_low',
 'undecided_over_high_+_low']

In [5]:
import nibabel
from joblib import Parallel, delayed


In [6]:
for contrast_type in contrast_list:
    subject_ids = {2, 4, 5, 6, 9, 11, 14, 15, 17, 19}
    subject_imgs = [f"../contrasts/smoothed_3mm/sub-{i}-{contrast_type}.nii.gz" for i in subject_ids]

    n_permutations = 10000

    n_subjects = len(subject_imgs)
    design_matrix = pd.DataFrame(
        [1] * n_subjects,
        columns=["intercept"]
    )

    second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
    second_level_model.fit(subject_imgs, design_matrix=design_matrix)

    observed_t_map = second_level_model.compute_contrast(
        second_level_contrast="intercept",
        output_type="stat"          # Returns t-statistic map
    )

    print("Observed t-map computed")
    print(f"Max t-value observed: {observed_t_map.get_fdata().max():.3f}")

    masker = NiftiMasker(
        mask_strategy="whole-brain-template",
        smoothing_fwhm=6.0,
        standardize=False
    )

    subject_data = masker.fit_transform(subject_imgs)
    # Shape: (n_subjects, n_voxels)
    print(f"Data shape: {subject_data.shape}")

    observed_masker = NiftiMasker(
        mask_strategy="whole-brain-template",
        standardize=False
    )

    observed_t_masked = observed_masker.fit_transform(observed_t_map).squeeze()

    def run_permutation_parallel(i, subject_data, design_matrix, masker):
        rng = np.random.RandomState(i)
        n_subjects = subject_data.shape[0]

        signs = rng.choice([-1, 1], size=n_subjects)
        permuted_data = subject_data * signs[:, np.newaxis]

        permuted_imgs = masker.inverse_transform(permuted_data)
        permuted_imgs_list = list(image.iter_img(permuted_imgs))

        perm_model = SecondLevelModel(smoothing_fwhm=None)
        perm_model.fit(permuted_imgs_list, design_matrix=design_matrix)

        perm_t_map = perm_model.compute_contrast(
            second_level_contrast="intercept",
            output_type="stat"
        )

        return masker.transform(perm_t_map).squeeze().max()


    # Run in parallel using all available cores
    null_distribution = Parallel(n_jobs=-1, verbose=10)(
        delayed(run_permutation_parallel)(
            i, subject_data, design_matrix, masker
        )
        for i in range(n_permutations)
    )

    null_distribution = np.array(null_distribution)
    # FWE-corrected threshold (95th percentile of null distribution)
    threshold_fwe_05 = np.percentile(null_distribution, 95)
    threshold_fwe_01 = np.percentile(null_distribution, 99)

    print(f"FWE-corrected threshold (p<0.05): t > {threshold_fwe_05:.3f}")
    print(f"FWE-corrected threshold (p<0.01): t > {threshold_fwe_01:.3f}")

    # Compute voxel-wise permutation p-values
    # For each voxel: proportion of permutations exceeding observed t
    p_values = np.array([
        np.mean(null_distribution >= t)
        for t in observed_t_masked
    ])

    # Convert to -log10(p) for visualization (same as nilearn output)
    # Add small epsilon to avoid log(0)
    log_p_map_data = -np.log10(p_values + 1e-10)

    # Reconstruct log-p image
    log_p_map = masker.inverse_transform(log_p_map_data)

    nibabel.save(log_p_map, f"../contrasts/group_p_values/{contrast_type}.nii.gz")
    np.save(f"../contrasts/group_p_values/{contrast_type}_null.npy", null_distribution)

    print(p_values.min(), p_values.max())
    print(log_p_map_data.min(), log_p_map_data.max())
    print(log_p_map_data.mean(), log_p_map_data.std())
    print(observed_t_masked.max())


Observed t-map computed
Max t-value observed: 5.311
Data shape: (10, 51075)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:   11.6s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:   14.8s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:   20.5s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:   23.8s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   26.9s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   30.1s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   36.0s
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed:   39.4s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:   44.8s
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed:   50.5s
[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed:   55.5s
[Parallel(n_jobs=-1)]: Done 129 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 165 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:  1

FWE-corrected threshold (p<0.05): t > 5.003
FWE-corrected threshold (p<0.01): t > 5.735
0.0241 1.0
-4.342945178152237e-11 1.6179829556230798
0.0031098483698400806 0.022071197734753127
5.310837732725797
Observed t-map computed
Max t-value observed: 3.981
Data shape: (10, 51075)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    5.6s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:   10.8s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.6s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   16.4s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   19.3s
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed:   24.7s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:   30.0s
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed:   35.4s
[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed:   38.4s
[Parallel(n_jobs=-1)]: Done 129 tasks      | elapsed:   46.2s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   51.5s
[Parallel(n_jobs=-1)]: Done 165 tasks      | elapsed:   57.5s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:  1

FWE-corrected threshold (p<0.05): t > 4.865
FWE-corrected threshold (p<0.01): t > 5.393
0.308 1.0
-4.342945178152237e-11 0.511449283358551
0.001723609066567008 0.01595631525849815
3.906077301998668
Observed t-map computed
Max t-value observed: 8.534
Data shape: (10, 51075)


...

FWE-corrected threshold (p<0.05): t > 4.146
FWE-corrected threshold (p<0.01): t > 4.526
0.0 1.0
-4.342945178152237e-11 10.0
0.00941646958114086 0.19795794426576968
6.334307812688353
Observed t-map computed
Max t-value observed: 7.702
Data shape: (10, 51075)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    5.4s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    5.5s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:   10.9s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.6s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   16.4s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   19.3s
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed:   24.7s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:   29.7s
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed:   35.0s
[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed:   38.7s
[Parallel(n_jobs=-1)]: Done 129 tasks      | elapsed:   45.9s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   51.4s
[Parallel(n_jobs=-1)]: Done 165 tasks      | elapsed:   57.3s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:  1

FWE-corrected threshold (p<0.05): t > 4.111
FWE-corrected threshold (p<0.01): t > 4.396
0.0 1.0
-4.342945178152237e-11 10.0
0.004024355301749623 0.13665848558549423
7.701919330294714
Observed t-map computed
Max t-value observed: 3.948
Data shape: (10, 51075)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    5.5s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.4s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   16.4s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   19.4s
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed:   24.6s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:   29.3s
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed:   34.5s
[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed:   38.8s
[Parallel(n_jobs=-1)]: Done 129 tasks      | elapsed:   45.1s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   50.4s
[Parallel(n_jobs=-1)]: Done 165 tasks      | elapsed:   57.6s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:  1

FWE-corrected threshold (p<0.05): t > 5.038
FWE-corrected threshold (p<0.01): t > 5.733
0.4071 1.0
-4.342945178152237e-11 0.39029889751392044
0.0006527729889602438 0.008717387232819535
3.560121328344705
